# CommitmentOS Checkpoint Evaluation (Colab)

This notebook compares a base model against a locally saved LoRA-trained checkpoint on the CommitmentOS environment.

It uses:
- `BASELINE_MODEL_NAME` from Hugging Face
- `TRAINED_MODEL_PATH` from disk in Colab
- the existing `evaluation/evaluate_llm_checkpoints.py` script

By default the notebook evaluates against the hosted CommitmentOS environment on Hugging Face Space. An optional local-server cell is included below.

In [ ]:
!pip -q install --upgrade pip
!pip -q install transformers peft accelerate torch sentencepiece fastapi uvicorn requests python-dotenv pydantic "openenv-core>=0.2.0"

In [ ]:
!git clone https://github.com/Jayant2304/commitment_os.git
%cd commitment_os

## Configure Paths

Set the base model ID and the local adapter/checkpoint path. Change `TRAINED_MODEL_PATH` to the folder you actually want to evaluate.

If the base model is gated, set `HF_TOKEN` as well.

In [ ]:
import os

# Colab: load Hugging Face token from Secrets (key must be exactly HF_TOKEN)
try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except ImportError:
    print("Not on Colab; set HF_TOKEN in the shell or .env if downloads fail.")
except Exception as exc:
    print("Could not load HF_TOKEN from secrets:", exc)

os.environ["BASELINE_MODEL_NAME"] = "Qwen/Qwen2.5-1.5B-Instruct"
os.environ["TRAINED_MODEL_PATH"] = "/content/commitment_os/training_output"
os.environ["ENV_BASE_URL"] = "https://jayant2304-commitment-os.hf.space"

# Optional for gated base models:
# os.environ["HF_TOKEN"] = "hf_xxx"

# Optional eval overrides:
os.environ["EVAL_SEED"] = "42"
os.environ["EVAL_MAX_STEPS"] = "12"
os.environ["EVAL_TEMPERATURE"] = "0.0"
os.environ["EVAL_TOP_P"] = "1.0"
os.environ["EVAL_MAX_NEW_TOKENS"] = "256"
os.environ["EVAL_SUCCESS_THRESHOLD"] = "0.6"

for key in [
    "BASELINE_MODEL_NAME",
    "TRAINED_MODEL_PATH",
    "ENV_BASE_URL",
    "EVAL_SEED",
    "EVAL_MAX_STEPS",
    "EVAL_TEMPERATURE",
    "EVAL_TOP_P",
    "EVAL_MAX_NEW_TOKENS",
    "EVAL_SUCCESS_THRESHOLD",
]:
    print(f"{key}={os.environ[key]}")

In [ ]:
from pathlib import Path

trained_path = Path(os.environ["TRAINED_MODEL_PATH"])
print("Checkpoint exists:", trained_path.exists())
if trained_path.exists():
    print("Checkpoint contents:")
    for item in sorted(trained_path.iterdir()):
        print(" -", item.name)

## Optional: Run CommitmentOS Locally Instead Of HF Space

Only run this if you want evaluation against a local server inside Colab. Otherwise skip this section and keep `ENV_BASE_URL` pointed at the hosted Space.

In [ ]:
# Optional local server setup
# import os
# os.environ["ENV_BASE_URL"] = "http://127.0.0.1:7860"
# !nohup python -m uvicorn server.app:app --host 0.0.0.0 --port 7860 >/tmp/commitmentos.log 2>&1 &
# !sleep 5
# !curl -s http://127.0.0.1:7860/health

## Run Checkpoint Comparison

In [ ]:
!python evaluation/evaluate_llm_checkpoints.py

In [ ]:
!python evaluation/plot_llm_checkpoints.py

## Inspect Artifacts

In [ ]:
import json
from pathlib import Path

artifact_dir = Path("artifacts/evals_llm")
print(sorted(p.name for p in artifact_dir.iterdir()))

summary = json.loads((artifact_dir / "llm_summary.json").read_text())
summary

In [ ]:
import pandas as pd

pd.read_csv("artifacts/evals_llm/llm_comparison.csv")

In [ ]:
from IPython.display import SVG, display

display(SVG(filename="artifacts/evals_llm/llm_reward_by_task.svg"))
display(SVG(filename="artifacts/evals_llm/llm_violations_before_after.svg"))

## Backup results (zip and download)

Run after eval/plot finish. Large runs: copy `training_output` to Google Drive instead of browser download.


In [ ]:
!cd /content/commitment_os && du -sh training_output artifacts/evals_llm 2>/dev/null || true
!cd /content/commitment_os && zip -r /content/commitment_os_bundle.zip training_output artifacts/evals_llm
from google.colab import files

files.download("/content/commitment_os_bundle.zip")
